# Tutorial 02: Pozo de Potencial Infinito (`InfiniteWellPotential`)

En este tutorial simularemos el confinamiento de un paquete de ondas relativista de Dirac dentro de una cavidad o pozo de potencial infinito en **1D y 2D** utilizando la clase `InfiniteWellPotential(widths=[...])`.

Formalmente, el potencial de caja centrada en el origen con anchos $(L_x, L_y, L_z)$ se define como:

$$V(\mathbf{r}) = \begin{cases} 0 & \text{si } |x_k| < \frac{L_k}{2} \;\; \forall k \in \{1, \dots, D\} \\ \infty & \text{en caso contrario} \end{cases}$$

### Objetivos de aprendizaje
1. Instanciar y visualizar `InfiniteWellPotential(widths=[L])` en 1D y `InfiniteWellPotential(widths=[Lx, Ly])` en 2D.
2. Observar las múltiples reflexiones del paquete de ondas en las paredes del pozo y los patrones de interferencia cuántica.
3. Monitorear la oscilación de la posición esperada $\langle x \rangle(t)$ y del momento esperado $\langle p_x \rangle(t)$ usando `Observables`.
4. Verificar cómo `Observables.expected_value_energy` maneja de forma segura regiones donde $V = \infty$ y $\psi = 0$ sin producir `NaN`.

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt

# Asegurar que dirac_solver sea localizable desde examples/, examples/es/ o examples/en/
for _rel in ['..', '../..']:
    _abs = os.path.abspath(_rel)
    if os.path.exists(os.path.join(_abs, 'dirac_solver')) and _abs not in sys.path:
        sys.path.insert(0, _abs)

from dirac_solver import (
    Grid,
    ConstantSpinor,
    GaussianPacket,
    PlaneWave,
    DiracProblemBuilder,
    DiracSolver,
    FreeParticle,
    ScalarPotential,
    CoulombPotential,
    YukawaPotential,
    InfiniteWellPotential,
    Observables,
    HDF5Storage,
    _core,
    electron_mass,
    c,
    hbar,
)
from dirac_solver.storage import create_animation, create_animation_1d

plt.rcParams.update({'figure.figsize': (10, 5), 'font.size': 12})

---
## 1. Confinamiento 1D en un Pozo de Potencial Infinito

Configuraremos un dominio 1D de longitud $20.0$ ($x \in [-10, 10]$) con un pozo infinito de ancho $L_x = 10.0$ ($x \in [-5, 5]$). Inicializaremos un paquete gaussiano en el centro ($x_0 = 0$) con momento $p_{0x} = 4.0$.

In [ ]:
# Malla 1D centrada en x = 0
grid_1d = Grid(shape=(300,), spacing=(0.08,))
x_coords = grid_1d.coords[:, 0]

# Pozo de potencial infinito de ancho L = 10.0 (paredes en x = -5 y x = +5)
well_width = 10.0
potential_well_1d = InfiniteWellPotential(widths=[well_width])

# Evaluamos el potencial sobre la malla para visualizar la región permitida
V_eval = np.array([potential_well_1d.evaluate([x]) for x in x_coords])
allowed_region = np.isfinite(V_eval)
print(f"Nombre del potencial C++: {potential_well_1d.get_name()}")
print(f"Puntos dentro del pozo (V = 0): {np.sum(allowed_region)} / {len(x_coords)}")

In [ ]:
# Estado inicial: paquete gaussiano localizado dentro del pozo
spinor_1d = ConstantSpinor(spin_polarization=[0, 0, 1], momentum=[4.0, 0.0, 0.0])
initial_packet_1d = GaussianPacket(spinor_1d, center=[0.0, 0.0, 0.0], spatial_width=0.9)

dt = 0.004
total_time = 6.0

problem_1d = (
    DiracProblemBuilder()
    .set_grid(grid_1d)
    .set_initial_state(initial_packet_1d)
    .set_potential(potential_well_1d)
    .set_boundary_condition("periodic")
    .set_time_parameters(time_step=dt, total_time=total_time)
    .build()
)

solver_1d = DiracSolver(problem_1d)
obs_1d = Observables(solver_1d)

# Avanzamos por etapas para capturar instantáneas de las reflexiones en las paredes
snapshots = [(0.0, obs_1d.probability_density(solver_1d.get_psi()).copy())]
obs_1d.compute_all(0.0)

num_steps = int(total_time / dt)
capture_steps = {int(0.25 * num_steps), int(0.5 * num_steps), int(0.75 * num_steps), num_steps}

for step in range(1, num_steps + 1):
    solver_1d.integrator.step()
    t_now = step * dt
    if step % 5 == 0 or step == num_steps:
        obs_1d.compute_all(t_now)
    if step in capture_steps:
        snapshots.append((t_now, obs_1d.probability_density(solver_1d.get_psi()).copy()))

print(f"Simulación completada. Instantáneas capturadas: {[round(t, 2) for t, _ in snapshots]}")

In [ ]:
# Graficar las instantáneas de la densidad de probabilidad dentro del pozo
fig, ax = plt.subplots(figsize=(11, 5))

for t_snap, rho_snap in snapshots:
    ax.plot(x_coords, rho_snap, lw=1.8, label=f't = {t_snap:.2f}')

# Sombrear las paredes infinitas (|x| >= 5.0)
ax.axvspan(x_coords[0], -well_width / 2, color='crimson', alpha=0.15, label='Paredes V = ∞')
ax.axvspan(well_width / 2, x_coords[-1], color='crimson', alpha=0.15)
ax.axvline(-well_width / 2, color='crimson', ls='--', lw=1.5)
ax.axvline(well_width / 2, color='crimson', ls='--', lw=1.5)

ax.set_xlabel('Posición x')
ax.set_ylabel(r'Densidad de probabilidad $\rho(x, t)$')
ax.set_title('Confinamiento y Reflexiones en un Pozo de Potencial Infinito 1D')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 2. Evolución de Valores Esperados $\langle x \rangle(t)$ y $\langle p_x \rangle(t)$

A medida que el paquete rebota entre $x = -5$ y $x = +5$, la posición esperada $\langle x \rangle(t)$ oscila periódicamente dentro de la cavidad mientras el momento medio $\langle p_x \rangle(t)$ invierte su signo en cada reflexión.

In [ ]:
times = obs_1d.get_history('time')
exp_x = obs_1d.get_history('expected_position')[:, 0]
exp_px = obs_1d.get_history('expected_momentum')[:, 0]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

ax1.plot(times, exp_x, 'b-', lw=2, label=r'$\langle x \rangle(t)$')
ax1.axhline(well_width / 2, color='r', ls='--', alpha=0.5, label='Fronteras del pozo')
ax1.axhline(-well_width / 2, color='r', ls='--', alpha=0.5)
ax1.set_ylabel(r'Posición esperada $\langle x \rangle$')
ax1.set_title('Rebotes del Paquete de Ondas en el Pozo Infinito')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(times, exp_px, 'darkorange', lw=2, label=r'$\langle p_x \rangle(t)$')
ax2.axhline(0.0, color='k', ls=':', alpha=0.5)
ax2.set_xlabel('Tiempo t')
ax2.set_ylabel(r'Momento esperado $\langle p_x \rangle$')
ax2.set_title('Inversión del Momento en cada Reflexión')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 3. Cavidad Bidimensional 2D (`InfiniteWellPotential(widths=[Lx, Ly])`)

Ahora confinaremos un paquete gaussiano 2D con momento oblicuo $\mathbf{p}_0 = (3.5, 2.0, 0)$ dentro de una caja cuadrada de dimensiones $L_x = 6.0, L_y = 6.0$.

In [ ]:
grid_2d = Grid(shape=(80, 80), spacing=(0.12, 0.12))
potential_well_2d = InfiniteWellPotential(widths=[6.0, 6.0])

spinor_2d = ConstantSpinor(spin_polarization=[0, 0, 1], momentum=[3.5, 2.0, 0.0])
initial_2d = GaussianPacket(spinor_2d, center=[0.0, 0.0, 0.0], spatial_width=0.8)

problem_2d = (
    DiracProblemBuilder()
    .set_grid(grid_2d)
    .set_initial_state(initial_2d)
    .set_potential(potential_well_2d)
    .set_boundary_condition("periodic")
    .set_time_parameters(time_step=0.005, total_time=1.8)
    .build()
)

solver_2d = DiracSolver(problem_2d)
rho_2d_init = np.sum(np.abs(solver_2d.get_psi())**2, axis=1).real.reshape(grid_2d.shape)

solver_2d.run_simulation()
rho_2d_final = np.sum(np.abs(solver_2d.get_psi())**2, axis=1).real.reshape(grid_2d.shape)

extent = [
    grid_2d.origin[0], grid_2d.origin[0] + (grid_2d.shape[0] - 1) * grid_2d.spacing[0],
    grid_2d.origin[1], grid_2d.origin[1] + (grid_2d.shape[1] - 1) * grid_2d.spacing[1],
]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, data, title in zip(axes, [rho_2d_init, rho_2d_final], ['Estado Inicial (t = 0)', 'Confinamiento 2D (t = 1.8)']):
    im = ax.imshow(data.T, origin='lower', extent=extent, cmap='magma')
    rect = plt.Rectangle((-3.0, -3.0), 6.0, 6.0, fill=False, edgecolor='cyan', ls='--', lw=2, label='Paredes del pozo')
    ax.add_patch(rect)
    ax.set_title(title)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    plt.colorbar(im, ax=ax, label=r'$\rho(x,y)$')

plt.tight_layout()
plt.show()